# 02c — AquaCrop (FAO), uncalibrated

AquaCrop-OSPy (package `aquacrop`), default crop `MaizeGDD`, ETo from BR-DWGD, same soil profile as APSIM (`notebooks/modelo_aquacrop.py`). Variables: simulated and potential yield, biomass, cumulative transpiration stress Σ(1 − Tr/TrPot) per period and minimum Tmin per period.

Part of the comparison of process-based models as feature generators for the hybrid model (MZA, AquaCrop,
APSIM, DSSAT). For a fair comparison, all use the same window of 4 sowing dates (15/01, 05/02, 25/02,
15/03) in the municipalities with second-season maize records in IBGE. Resumes where it stopped (one part per municipality).

In [1]:
from pathlib import Path

import pandas as pd

import rodar_lote

RAIZ = Path.cwd().parent
DADOS, RESULTADOS = RAIZ / "dados", RAIZ / "resultados"
MODELO = "aquacrop"
DATAS = [(1, 15), (2, 5), (2, 25), (3, 15)]
PARTES = RESULTADOS / f"{MODELO}_partes"

pam = pd.read_csv(DADOS / "ibge_pam_milho_pr.csv")
codigos = sorted(pam.loc[(pam["safra"] == 2) & (pam["rendimento_kg_ha"] > 0) & (pam["area_colhida_ha"] >= 100),
                         "cod_ibge"].unique())
print(len(codigos), "municipalities")
falhas = rodar_lote.rodar(MODELO, codigos, DATAS, PARTES, processos=4)
print("failures:", len(falhas), list(falhas.items())[:3])

373 municípios


falhas: 0 []


In [2]:
sim = pd.concat([pd.read_parquet(p) for p in sorted(PARTES.glob("*.parquet"))], ignore_index=True)
sim.to_parquet(RESULTADOS / f"simulacao_{MODELO}_pr.parquet", index=False)
print(sim.shape, sim["cod_ibge"].nunique(), "municipalities")
sim.groupby("semeadura").mean(numeric_only=True).drop(columns=["cod_ibge", "ano"]).round(2).T

(40284, 15) 373 municípios


semeadura,05/02,15/01,15/03,25/02
aq_rend,2173.48,1869.47,1990.25,1686.88
aq_rend_pot,3793.06,3182.71,2893.29,2644.60
aq_biomassa,707.06,770.50,807.01,637.59
aq_cc_max,0.48,0.56,0.57,0.49
aq_ciclo_dias,112.32,102.33,136.80,124.61
aq_razao_pot,0.53,0.54,0.67,0.64
aq_estresse_veg,3.90,3.25,3.37,3.01
aq_tmin_veg,14.96,15.87,8.38,12.26
aq_estresse_flor,1.81,1.76,1.89,1.74
aq_tmin_flor,14.60,16.81,7.73,10.82
